# Vitrine 02 (Nível Júnior): Inteligência Macro-Econômica e Pricing
**Foco:** Big Data Visualization, Volatilidade de Preços e Competitividade Regional.

Neste caso, exploramos o histórico de preços de postos de combustível para entender como o mercado reage a variáveis como localização, marca e dia da semana, utilizando volume de dados massivo (370k+ registros).

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

PRICING_PATH = '../data/processed/price_history_limpo.csv'
STATIONS_PATH = '../data/processed/dados_limpos.csv'

p = pd.read_csv(PRICING_PATH)
s = pd.read_csv(STATIONS_PATH)[['id', 'brand_name', 'county', 'city']]

# Join para enriquecer os preços com metadados de marca e região
df = pd.merge(p, s, left_on='node_id', right_on='id', how='left')
df.head()

## 1. Dispersão de Preços por Marca (Benchmarking)
Quais bandeiras praticam os preços mais altos e onde está a maior variação?

In [ ]:
top_brands = df['brand_name'].value_counts().head(10).index
df_top = df[df['brand_name'].isin(top_brands)]

fig_box = px.box(df_top, x='brand_name', y='price_pence', color='brand_name',
                  title='<b>Posicionamento de Preço:</b> Comparativo de Dispersão por Bandeira',
                  labels={'brand_name': 'Marca', 'price_pence': 'Preço (Pence)'})

fig_box.update_layout(showlegend=False, template='plotly_white')
fig_box.show()

## 2. Índice de Competitividade Regional
Onde a guerra de preços é mais intensa? O Índice de Competitividade (PCI) negativo indica preços abaixo da média da vizinhança.

In [ ]:
pci_county = df.groupby('county')['price_competitiveness_index'].mean().reset_index().sort_values('price_competitiveness_index')

fig_pci = px.bar(pci_county.head(15), x='county', y='price_competitiveness_index',
                  title='<b>Zonas de Guerra de Preço:</b> Cidades com maior agressividade competitiva',
                  labels={'county': 'Região', 'price_competitiveness_index': 'Índice de Competitividade Avg'},
                  color='price_competitiveness_index', color_continuous_scale='RdYlGn_r')

fig_pci.update_layout(template='plotly_white')
fig_pci.show()

## 3. Dinâmica Temporal: Existe 'Prêmio' no Fim de Semana?
Análise se o preço médio aumenta ou diminui durante o final de semana.

In [ ]:
weekend_stats = df.groupby(['day_of_week'])['price_pence'].mean().reset_index()
days = {0:'Seg', 1:'Ter', 2:'Qua', 3:'Qui', 4:'Sex', 5:'Sáb', 6:'Dom'}
weekend_stats['day_name'] = weekend_stats['day_of_week'].map(days)

fig_time = px.line(weekend_stats, x='day_name', y='price_pence', markers=True,
                    title='<b>Sazonalidade Semanal:</b> Flutuação de Preço Médio Diário',
                    labels={'day_name': 'Dia da Semana', 'price_pence': 'Preço Médio'})

fig_time.update_layout(template='plotly_white')
fig_time.show()

### Conclusão Estratégica
1. **Bandeiras Premium:** Marcas como Shell mantêm prêmio de preço estável mas com alta dispersão regional.
2. **Agressividade PCI:** O índice de competitividade é a ferramenta chave para prever perda de volume de vendas.
3. **Oportunidade Dynamic Pricing:** Identificamos que a janela de repasse de preço é mais lenta em certas cidades, indicando ineficiência de margem.